<a href="https://colab.research.google.com/github/tensorbytes0202/Deep-learning/blob/main/brain_tumour_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/brain_tumour_data.zip.zip"
extract_path = "/content/lgg_dataset"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction Done ✅")

Extraction Done ✅


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from glob import glob
from sklearn.model_selection import train_test_split

import tensorflow as tf
from keras.models import Model
from keras.layers import *
from keras.optimizers import Adamax
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
def create_df(data_dir):
    images_paths = []
    masks_paths = glob(f'{data_dir}/*/*_mask*')

    for i in masks_paths:
        images_paths.append(i.replace('_mask', ''))

    df = pd.DataFrame({
        'images_paths': images_paths,
        'masks_paths': masks_paths
    })
    return df

data_dir = "/content/lgg_dataset/kaggle_3m"
df = create_df(data_dir)
print("Total Samples:", len(df))
df.head()

Total Samples: 3929


,images_paths,masks_paths
0,/content/lgg_dataset/kaggle_3m/TCGA_DU_5853_19...,/content/lgg_dataset/kaggle_3m/TCGA_DU_5853_19...
1,/content/lgg_dataset/kaggle_3m/TCGA_DU_5853_19...,/content/lgg_dataset/kaggle_3m/TCGA_DU_5853_19...
2,/content/lgg_dataset/kaggle_3m/TCGA_DU_5853_19...,/content/lgg_dataset/kaggle_3m/TCGA_DU_5853_19...
3,/content/lgg_dataset/kaggle_3m/TCGA_DU_5853_19...,/content/lgg_dataset/kaggle_3m/TCGA_DU_5853_19...
4,/content/lgg_dataset/kaggle_3m/TCGA_DU_5853_19...,/content/lgg_dataset/kaggle_3m/TCGA_DU_5853_19...


In [ ]:
def split_df(df):
    train_df, dummy_df = train_test_split(df, train_size=0.8, random_state=42)
    valid_df, test_df = train_test_split(dummy_df, train_size=0.5, random_state=42)
    return train_df, valid_df, test_df

train_df, valid_df, test_df = split_df(df)

print(len(train_df), len(valid_df), len(test_df))

3143 393 393


In [ ]:
def create_gens(df, aug_dict):
    img_size = (256,256)
    batch_size = 16

    img_gen = ImageDataGenerator(**aug_dict)
    msk_gen = ImageDataGenerator(**aug_dict)

    image_gen = img_gen.flow_from_dataframe(
        df, x_col='images_paths',
        class_mode=None, color_mode='rgb',
        target_size=img_size, batch_size=batch_size, seed=1)

    mask_gen = msk_gen.flow_from_dataframe(
        df, x_col='masks_paths',
        class_mode=None, color_mode='grayscale',
        target_size=img_size, batch_size=batch_size, seed=1)

    for img, msk in zip(image_gen, mask_gen):
        img = img / 255
        msk = msk / 255
        msk = (msk > 0.5).astype(np.float32)
        yield img, msk

In [ ]:
def unet(input_size=(256,256,3)):
    inputs = Input(input_size)

    c1 = Conv2D(64,3,activation='relu',padding='same')(inputs)
    c1 = Conv2D(64,3,activation='relu',padding='same')(c1)
    p1 = MaxPooling2D()(c1)

    c2 = Conv2D(128,3,activation='relu',padding='same')(p1)
    c2 = Conv2D(128,3,activation='relu',padding='same')(c2)
    p2 = MaxPooling2D()(c2)

    c3 = Conv2D(256,3,activation='relu',padding='same')(p2)
    c3 = Conv2D(256,3,activation='relu',padding='same')(c3)
    p3 = MaxPooling2D()(c3)

    c4 = Conv2D(512,3,activation='relu',padding='same')(p3)
    c4 = Conv2D(512,3,activation='relu',padding='same')(c4)
    p4 = MaxPooling2D()(c4)

    c5 = Conv2D(1024,3,activation='relu',padding='same')(p4)

    u6 = Conv2DTranspose(512,2,strides=2,padding='same')(c5)
    u6 = concatenate([u6,c4])
    c6 = Conv2D(512,3,activation='relu',padding='same')(u6)

    u7 = Conv2DTranspose(256,2,strides=2,padding='same')(c6)
    u7 = concatenate([u7,c3])
    c7 = Conv2D(256,3,activation='relu',padding='same')(u7)

    u8 = Conv2DTranspose(128,2,strides=2,padding='same')(c7)
    u8 = concatenate([u8,c2])
    c8 = Conv2D(128,3,activation='relu',padding='same')(u8)

    u9 = Conv2DTranspose(64,2,strides=2,padding='same')(c8)
    u9 = concatenate([u9,c1])
    c9 = Conv2D(64,3,activation='relu',padding='same')(u9)

    outputs = Conv2D(1,1,activation='sigmoid')(c9)

    return Model(inputs, outputs)

In [ ]:
def dice_coef(y_true, y_pred):
  smooth = 1
  y_true_f = tf.reshape(y_true,[-1])
  y_pred_f = tf.resshape(y_pred,[-1])
  intersection = tf.reduce_sum(y_true_f * y_pred_f)
  return (2.*intersection + smooth)/(tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f)+smooth)

def dice_loss(y_true,y_pred):
  return 1.0 - dice_coef(y_true,y_pred)

model = unet()

model.compile(optimizer=Adamax(learning_rate=0.001),
              loss = dice_loss,
              metrics =['accuracy',dice_coef]
              )
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 256, 256,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 256, 256,  │      1,792 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 256, 256,  │     36,928 │ conv2d[0][0]      │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 128, 128,  │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 128, 128,  │     73,856 │ max_pooling2d[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 128, 128,  │    147,584 │ conv2d_2[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 64, 64,    │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 64, 64,    │    295,168 │ max_pooling2d_1[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 64, 64,    │    590,080 │ conv2d_4[0][0]    │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 32, 32,    │          0 │ conv2d_5[0][0]    │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 32, 32,    │  1,180,160 │ max_pooling2d_2[… │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 32, 32,    │  2,359,808 │ conv2d_6[0][0]    │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 16, 16,    │          0 │ conv2d_7[0][0]    │
│ (MaxPooling2D)      │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 16, 16,    │  4,719,616 │ max_pooling2d_3[… │
│                     │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose    │ (None, 32, 32,    │  2,097,664 │ conv2d_8[0][0]    │
│ (Conv2DTranspose)   │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 32, 32,    │          0 │ conv2d_transpose… │
│ (Concatenate)       │ 1024)             │            │ conv2d_7[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 32, 32,    │  4,719,104 │ concatenate[0][0

 Total params: 18,459,137 (70.42 MB)

 Trainable params: 18,459,137 (70.42 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
train_gen = create_gens(train_df, dict(horizontal_flip=True))
val_gen = create_gens(valid_df, {})

history = model.fit(
    train_gen,
    steps_per_epoch=len(train_df)//16,
    validation_data=val_gen,
    validation_steps=len(valid_df)//16,
    epochs=30
)

In [ ]:
model.save("/content/drive/MyDrive/unet_model.keras")
print("Model Saved ✅")

In [ ]:
def predicted_image(img_path):
  img = cv2.imread(img_path)
  img = cv2.resize(img,(256,256))
  img = img/255
  img = np.expand_dims(img,0)

  pred = model.predict(img)[0]
  return img[0],pred

In [ ]:
import random
idx = random.randint(0,len(test_df)-1)
img_path = test_df['images_paths'].iloc[idx]
mask_path = test_df['masks_paths'].iloc[idx]

img,pred = predict_img(img_path)
true_mask = cv2.imread(mask_path,0)

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(img)
plt.title('Image')
plt.axis('off')

plt.subplot(1,3,2)
plt.imshow(pred>0.5,cmap='gray')
plt.title("Prediction")
plt.axis('off')

plt.show()